In [1]:
# library imports
import numpy as np
import pandas as pd
from gluonts.dataset.common import ListDataset
from gluonts.mx import DeepVAREstimator
from gluonts.mx.distribution import MultivariateGaussianOutput
from gluonts.mx import Trainer

# just for jupyter notebooks plot size
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

/Users/sermetpekin/Desktop/git_repos/nowcasting_benchmark/.venv/lib/python3.11/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


OSError: dlopen(/Users/sermetpekin/Desktop/git_repos/nowcasting_benchmark/.venv/lib/python3.11/site-packages/mxnet/libmxnet.so, 0x0006): tried: '/Users/sermetpekin/Desktop/git_repos/nowcasting_benchmark/.venv/lib/python3.11/site-packages/mxnet/libmxnet.so' (slice is not valid mach-o file), '/System/Volumes/Preboot/Cryptexes/OS/Users/sermetpekin/Desktop/git_repos/nowcasting_benchmark/.venv/lib/python3.11/site-packages/mxnet/libmxnet.so' (no such file), '/Users/sermetpekin/Desktop/git_repos/nowcasting_benchmark/.venv/lib/python3.11/site-packages/mxnet/libmxnet.so' (slice is not valid mach-o file)

In [ ]:
# helper function, generate lagged datasets for testing on vintages
def gen_lagged_data(metadata, data, last_date, lag):
    # only go up to the last date
    lagged_data = data.loc[data.date <= last_date, :].reset_index(drop=True)
    for col in lagged_data.columns[1:]:
        pub_lag = metadata.loc[metadata.series == col, "months_lag"].values[0] # publication lag of this particular variable
        # go back as far as needed for the pub_lag of the variable, then + the lag (so -2 for 2 months back), also -1 because 0 lag means in month, last month data available, not current month in
        lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1) :, col] = np.nan

    return lagged_data

# helper function, flatten a dataset for methods that don't do timeseries, extra columns for each lag
def flatten_data(data, target_variable, n_lags):
    flattened_data = data.loc[~pd.isna(data[target_variable]), :]
    orig_index = flattened_data.index
    for i in range(1, n_lags + 1):
        lagged_indices = orig_index - i
        lagged_indices = lagged_indices[lagged_indices >= 0]
        tmp = data.loc[lagged_indices, :]
        tmp.date = tmp.date + pd.DateOffset(months=i)
        tmp = tmp.drop([target_variable], axis=1)
        tmp.columns = [j + "_" + str(i) if j != "date" else j for j in tmp.columns]
        flattened_data = flattened_data.merge(tmp, how="left", on="date")

    return flattened_data

# helper function fill missings in a dataset with the mean from the training set
def mean_fill_dataset(training, test):
    mean_dict = {}
    for col in training.columns[1:]:
        mean_dict[col] = np.nanmean(training[col])
    filled = test.copy()
    for col in training.columns[1:]:
        filled.loc[pd.isna(filled[col]), col] = mean_dict[col]
    return filled

# Data set up

In [ ]:
# data read
data = pd.read_csv("../data/data_tf.csv", parse_dates=["date"])
metadata = pd.read_csv("../data/meta_data.csv")

# target variable = GDP
target_variable = "gdpc1"

# which artificial lags to test the model on, equivalent to number of months ahead of and behind the target date (e.g. 2020-06-01 for Q2 2020)
lags = list(range(-2, 3))

# train and test dates
train_start_date = "1947-01-01"
test_start_date = "2005-03-01"
test_end_date = "2010-03-01"

# train and test datasets
test = data.loc[(data.date >= train_start_date) & (data.date <= test_end_date), :].reset_index(drop=True)

Data should be a dataframe of seasonally adjusted growth rates with months in rows and quarterly variables in the last month of the quarter, with `np.nan`s for interquarter months.

In [ ]:
data.tail()

# Training the model

Because of how DeepVAR models are estimated, similar to ARMA models, there is no model training per se, the model is fitted and n-step ahead forecasts are generated at each data vintage in the testing set.

# Testing the model on artificial data vintages

In [ ]:
# dates in the test set
dates = (
        pd.date_range(test_start_date, test_end_date, freq="3MS")
        .strftime("%Y-%m-%d")
        .tolist()
    )

# actual values
actuals = list(test.loc[test.date.isin(dates), target_variable].values)

In [ ]:
pred_dict = {k: [] for k in lags}
for date in dates:
    # training the actual model
    train = test.loc[test.date <= str(pd.to_datetime(date) - pd.tseries.offsets.DateOffset(months=3))[:10],:] # data as it would have appeared at beginning of prediction period
    # lead the target variable 1 forward so that 1 ahead forecast is forecasting the target month
    train[target_variable] = train[target_variable].shift(1)
    
    # adjustments to estimate a DeepVAR model
    trainingset = train.copy()
    trainingset = mean_fill_dataset(train, trainingset)
    trainingset.set_index("date",drop=True, inplace=True)
    start = pd.Period(train.iloc[0,0], freq="M")
    
    training_data = ListDataset(
        [{"target": np.transpose(trainingset.values), "start": start}],
        freq = "M",
        one_dim_target = False
    )
    
    # model
    estimator = DeepVAREstimator(freq='M',
        target_dim = len(train.columns) - 1, # - 1 for date column 
        prediction_length = 1,
        context_length = 12, # number of trailing months to consider
        num_layers = 4,
        num_cells = 40,
        cell_type = 'lstm',
        distr_output = MultivariateGaussianOutput(dim = len(train.columns) - 1),
        dropout_rate = 1e-2, 
        trainer = Trainer(
            epochs = 10, 
            learning_rate = 1e-4
        ),
        batch_size = 100
    )
        
    predictor = estimator.train(training_data)
    
    
    for lag in lags:
        # the data available for this date at this artificial vintage
        tmp_data = gen_lagged_data(metadata, test, date, lag)

        # get data in format necessary for model
        tmp_data = mean_fill_dataset(train, tmp_data)
        tmp_data.set_index("date", drop=True, inplace=True)
        
        test_data = ListDataset(
            [{"target": np.transpose(tmp_data.values), "start": pd.Period(test.iloc[0,0], freq="M")}],
            freq = "M",
            one_dim_target = False
        )
            
        # predictions
        pred_obj = list(predictor.predict(test_data))
        pred = pred_obj[0].mean[0][list(train.columns)[1:].index(target_variable)] # mean[0] for 1 step ahead, last [] for index of target variable
        
        pred_dict[lag].append(pred)

# Assess and visualize model performance

In [ ]:
# table of RMSE by vintage
performance = pd.DataFrame(columns=["Vintage", "RMSE"])
for lag in lags:
    tmp = pd.DataFrame({
        "Vintage":lag,
        "RMSE":np.sqrt(np.mean((np.array(actuals) - np.array(pred_dict[lag])) ** 2))
    }, index=[0])
    performance = pd.concat([performance, tmp]).reset_index(drop=True)
performance.round(4)

In [ ]:
# plot of predictions vs actuals
pd.DataFrame({
    "actuals":actuals, 
    "two_back":pred_dict[-2], 
    "one_back":pred_dict[-1], 
    "zero_back":pred_dict[0],
    "one_ahead":pred_dict[1],
    "two_ahead":pred_dict[2]}
).plot()
;

# Final model usage / getting predictions on new data
Say model selection is finished and the model is to now be used, i.e. used to get predictions on new data.

In [ ]:
# the test data ends 2010-03-01, let's say we wanted to predict 2010-06-01
new_data = test.copy()

# the date we want predicted must be in the date, if it's not there it must be added
desired_date = pd.to_datetime("2010-06-01")

while desired_date > np.max(new_data.date):
    new_data.loc[len(new_data), "date"] = np.max(new_data.date) + pd.DateOffset(months=1)

# we can now confirm the date we want to forecast is in the dataframe, even if all values are missing
new_data.tail()

In [ ]:
# now transform the data into the appropriate format for the model
latest_data = test.copy()
latest_data[target_variable] = latest_data[target_variable].shift(1)

latest_dataset = latest_data.copy()
latest_dataset = mean_fill_dataset(latest_data, latest_dataset)
latest_dataset.set_index("date",drop=True, inplace=True)
start = pd.Period(latest_data.iloc[0,0], freq="M")

training_dataset = ListDataset(
    [{"target": np.transpose(latest_dataset.values), "start": start}],
    freq = "M",
    one_dim_target = False
)

In [ ]:
# training the model
estimator = DeepVAREstimator(freq='M',
    target_dim = len(latest_data.columns) - 1, # - 1 for date column 
    prediction_length = 1,
    context_length = 12, # number of trailing months to consider
    num_layers = 4,
    num_cells = 40,
    cell_type = 'lstm',
    distr_output = MultivariateGaussianOutput(dim = len(latest_data.columns) - 1),
    dropout_rate = 1e-2, 
    trainer = Trainer(
        epochs = 10, 
        learning_rate = 1e-4
    ),
    batch_size = 100
)

predictor = estimator.train(training_dataset)

In [ ]:
# obtain prediction for the new period
pred_obj = list(predictor.predict(training_dataset))
pred = pred_obj[0].mean[0][list(train.columns)[1:].index(target_variable)] # mean[0] for 1 step ahead, last [] for index of target variable
pred